# Решения: практикум CI/correlation

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('startup_ab.csv')
df = pd.read_csv(CSV_PATH)
df['variant_b'] = (df['variant'] == 'B').astype(int)


In [ ]:
seg = (
    df.groupby(['traffic_source', 'device'])
    .agg(n=('user_id', 'count'), conv=('converted', 'mean'))
    .reset_index()
)
def ci_uplift(part, seed=0, n_iter=1500):
    rng = np.random.default_rng(seed)
    a = part[part['variant'] == 'A']['converted'].to_numpy()
    b = part[part['variant'] == 'B']['converted'].to_numpy()
    if len(a) < 20 or len(b) < 20:
        return np.nan, np.nan
    vals = []
    for _ in range(n_iter):
        vals.append(float(rng.choice(b, len(b), replace=True).mean() - rng.choice(a, len(a), replace=True).mean()))
    return tuple(np.quantile(vals, [0.025, 0.975]))
rows = []
for i, src in enumerate(sorted(df['traffic_source'].unique())):
    part = df[df['traffic_source'] == src]
    low, high = ci_uplift(part, seed=390 + i)
    rows.append({'traffic_source': src, 'ci_low': low, 'ci_high': high})
ci_source = pd.DataFrame(rows)
SIGN_NOTE = (
    'В отдельных сегментах знак может отличаться из-за шума и разных размеров подвыборок. '
    'Поэтому важны интервалы и проверка устойчивости эффекта.'
)
corr_age = float(df['age'].corr(df['converted']))
REPORT_LINE = (
    'По сегментам видно, что эффект B не обязан быть одинаковым во всех каналах: '
    'часть различий может быть статистическим шумом в малых группах.'
)
rows2 = []
for d, part in df.groupby('discount_pct'):
    low, high = ci_uplift(part, seed=410 + int(d))
    rows2.append({'discount_pct': int(d), 'ci_low': low, 'ci_high': high})
ci_discount = pd.DataFrame(rows2).sort_values('discount_pct')
corr_prior = float(df['prior_visits_30d'].corr(df['converted']))
STABILITY_NOTE = (
    'Эффект устойчивее, когда знак uplift совпадает в ключевых сегментах и интервалы не слишком широкие. '
    'Если интервалы широкие, нужен больший объём данных.'
)
CHECKLIST = (
    'Перед product-решением проверяем: протокол эксперимента, размер выборки, CI эффекта, '
    'чувствительность к сегментам, отсутствие peeking и согласованность с бизнес-ограничениями.'
)
print(seg.head())
print(ci_source)
print(corr_age, corr_prior)